# Phase 1.2：从字符串切片手写 Chunking

## 目标

先不用项目里的 `RecursiveSplitter`，手写一个最小可理解版本。通过小字符串观察：长度上限、overlap、分隔符和边界错误分别是什么，再接入正式实现。

**本课交付：** `data/processed/phase1_chunk_experiment.json`。

## Evidence Quest 任务卡：Phase 1.2：边界救援任务

**你的身份：** 证据切片工程师  
**案件背景：** 一条关键线索正好卡在两个 Chunk 的边界上。切得太大难搜索，切得太小又会把上下文拆散。

### 本关专业 Goal

用实验理解 chunk_size、overlap 和分隔符如何决定证据边界。

### 你要交付的作品

**Chunk 边界实验板 + 边界救援结论**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：边界救援员  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 最小问题：如何把长文本变短？

Python 字符串可以用切片 `text[start:end]` 取出一段。`start` 包含在结果中，`end` 不包含在结果中。我们先用固定窗口切，不考虑语义；这是理解算法的地基。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase1.2'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase1.2
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 创建一段超过窗口长度的教学文本。
text = "ABCDEFGHIJ" * 4

# 设置每个窗口最多保存 10 个字符。
chunk_size = 10

# 从第 0 个字符开始取出第一个窗口。
first_chunk = text[0:chunk_size]

# 打印原文和第一个窗口，观察切片的边界。
print("原文:", text)
print("第一个 Chunk:", first_chunk)

# 验证窗口长度没有超过上限。
assert len(first_chunk) == chunk_size

原文: ABCDEFGHIJABCDEFGHIJABCDEFGHIJABCDEFGHIJ
第一个 Chunk: ABCDEFGHIJ


In [4]:
# 创建空列表，用于保存固定窗口切分结果。
simple_chunks = []

# 每次向前移动 chunk_size 个字符，直到走完整个文本。
for start in range(0, len(text), chunk_size):
    # 根据当前起点取出一个不超过上限的窗口。
    chunk = text[start : start + chunk_size]

    # 把窗口追加到结果列表。
    simple_chunks.append(chunk)

# 打印所有窗口，观察长文本如何被切成多个独立片段。
print(simple_chunks)

# 验证每个窗口都满足长度上限。
assert all(len(chunk) <= chunk_size for chunk in simple_chunks)

['ABCDEFGHIJ', 'ABCDEFGHIJ', 'ABCDEFGHIJ', 'ABCDEFGHIJ']


## 2. 加入 overlap：为什么窗口不再只向前移动 `chunk_size`？

如果相邻 Chunk 之间没有共享内容，事实刚好在边界断开时，两个 Chunk 都可能只拿到一半线索。设 `overlap=3`，下一块的起点就从 `start + chunk_size - overlap` 开始。

注意：overlap 复制的是上下文，不是新增信息。它会增加 Chunk 数或重复内容，因此必须通过检索评估决定是否值得。

In [5]:
# 设置相邻 Chunk 共享的字符数。
overlap = 3

# 计算相邻窗口真正需要移动的步长。
step = chunk_size - overlap

# 确保步长为正数，否则循环不会向前推进。
assert step > 0

# 创建空列表，用于保存带 overlap 的窗口。
overlapping_chunks = []

# 按 step 移动起点，让相邻窗口发生重叠。
for start in range(0, len(text), step):
    # 取出当前窗口。
    chunk = text[start : start + chunk_size]

    # 保存当前窗口。
    overlapping_chunks.append(chunk)

# 打印窗口，观察相邻结果的重复部分。
for index, chunk in enumerate(overlapping_chunks):
    # 输出编号和正文，方便肉眼比较边界。
    print(index, chunk)

# 验证所有窗口仍然遵守最大长度。
assert all(len(chunk) <= chunk_size for chunk in overlapping_chunks)

0 ABCDEFGHIJ
1 HIJABCDEFG
2 EFGHIJABCD
3 BCDEFGHIJA
4 IJABCDEFGH
5 FGHIJ


### 一个必须理解的边界错误

当 `overlap >= chunk_size` 时，`step <= 0`，起点不会正常前进。生产实现必须在初始化时拒绝这个参数，而不是等循环卡死。参数校验是算法正确性的一部分。

In [6]:
# 导入项目中的正式分块器。
from phase1_doc_parser.splitter import RecursiveSplitter

# 用非法参数创建分块器，验证正式实现会主动拒绝。
try:
    # overlap 等于 chunk_size，意味着窗口没有可移动空间。
    RecursiveSplitter(chunk_size=10, overlap=10)
except ValueError as error:
    # 打印清晰的错误信息，理解失败原因。
    print("参数被拒绝:", error)

参数被拒绝: overlap must be in [0, chunk_size)


## 3. 为什么要优先按段落和标点切？

固定字符窗口简单但可能把一句话切成两半。Recursive Splitter 的思想是：先尝试段落边界，再尝试换行和标点；只有某一段仍然太长时，才降级到更细的分隔符。它在“语义完整”和“长度上限”之间做折中。

In [7]:
# 定义一段包含段落、换行和中文标点的文本。
structured_text = "第一段介绍来源。这里还有补充。\n\n第二段介绍 overlap；它保留边界上下文。"

# 使用小窗口强迫分块器展示边界选择过程。
splitter = RecursiveSplitter(chunk_size=24, overlap=6)

# 执行正式分块。
recursive_chunks = splitter.split(structured_text)

# 逐个打印 Chunk 及字符长度。
for index, chunk in enumerate(recursive_chunks):
    # 输出编号、长度和内容，观察自然边界是否被保留。
    print(index, len(chunk), repr(chunk))

# 验证正式结果也满足长度约束。
assert recursive_chunks
assert all(len(chunk) <= 24 for chunk in recursive_chunks)

0 15 '第一段介绍来源。这里还有补充。'
1 23 '第二段介绍 overlap；它保留边界上下文。'


## 4. 比较参数，而不是凭感觉选参数

下面只改变 `chunk_size` 和 `overlap`，记录 Chunk 数、平均长度和最大长度。注意：这些是数据形状指标，不是检索质量指标；Phase 2 会用 qrels 检查哪个配置更容易召回正确证据。

In [8]:
# 定义待比较的参数组合。
configurations = [(32, 0), (32, 8), (64, 16), (128, 32)]

# 创建空列表，用于保存每组配置的统计结果。
experiment_rows = []

# 逐组运行分块实验。
for current_size, current_overlap in configurations:
    # 用当前参数创建一个新的分块器。
    current_splitter = RecursiveSplitter(chunk_size=current_size, overlap=current_overlap)

    # 对真实教学文本进行分块。
    current_chunks = current_splitter.split(structured_text)

    # 收集每个 Chunk 的字符长度。
    current_lengths = [len(chunk) for chunk in current_chunks]

    # 计算并记录可比较的数据形状指标。
    experiment_rows.append({"chunk_size": current_size, "overlap": current_overlap, "count": len(current_chunks), "avg_chars": round(sum(current_lengths) / len(current_lengths), 2), "max_chars": max(current_lengths)})

# 输出实验结果，先观察参数改变带来的现象。
for row in experiment_rows:
    # 每次打印一组配置和它的统计结果。
    print(row)

# 确认每组结果都满足最大长度约束。
assert all(row["max_chars"] <= row["chunk_size"] for row in experiment_rows)

{'chunk_size': 32, 'overlap': 0, 'count': 2, 'avg_chars': 19.0, 'max_chars': 23}
{'chunk_size': 32, 'overlap': 8, 'count': 2, 'avg_chars': 23.0, 'max_chars': 31}
{'chunk_size': 64, 'overlap': 16, 'count': 1, 'avg_chars': 40.0, 'max_chars': 40}
{'chunk_size': 128, 'overlap': 32, 'count': 1, 'avg_chars': 40.0, 'max_chars': 40}


In [9]:
# 创建处理数据目录，保证实验记录有固定位置。
experiment_directory = ROOT / "data" / "processed"
experiment_directory.mkdir(parents=True, exist_ok=True)

# 指定本课实验记录路径。
experiment_path = experiment_directory / "phase1_chunk_experiment.json"

# 把实验参数和结果一起保存，避免只保留结论而丢失条件。
experiment_path.write_text(json.dumps(experiment_rows, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印实验记录路径。
print("已生成:", experiment_path)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\phase1_chunk_experiment.json


## 本课验收

- [ ] 能解释 `text[start:end]` 的边界。
- [ ] 能推导 `step = chunk_size - overlap`。
- [ ] 能说出 overlap 的收益和成本。
- [ ] 能解释为什么递归分隔符优先于固定字符切片。
- [ ] 已保存 `phase1_chunk_experiment.json`。

下一课把这些理解接到批量构建器，正式生成整个项目后续要用的 `chunks.json`。

## Boss Challenge：只把 overlap 改成另一个值，预测 Chunk 数、边界和重复文本会如何变化。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [10]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [11]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase1_chunk_experiment.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase1_chunk_experiment.json']
